# Benchmark: THP vs RoTHP vs HoTHP em Datasets Reais

**Objetivo:** Comparar os três modelos nos 5 datasets HuggingFace do EasyTPP,
com 3 seeds, avaliando:
- **NLL** (Negative Log-Likelihood)
- **Accuracy** (tipo de evento)
- **RMSE** (tempo inter-evento)

Para datasets com sequências longas o suficiente (retweet, stackoverflow, amazon),
treina com sequências truncadas e avalia em 1× (truncado) e 5× (completo) — **extrapolação real**.

Para datasets com sequências curtas (taxi, earthquake), avalia apenas no tamanho natural.

**Progresso é salvo em pickle** após cada (dataset, modelo, seed) para sobreviver a quedas de conexão.

**Run on Colab:** Runtime → Change runtime type → T4 GPU

In [2]:
import os, sys

# ── Clona o repositório ufc-easytpp ───────────────────────────────────
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

sys.path.insert(0, 'ufc-easytpp')

# ── Instala dependências ──────────────────────────────────────────────
!pip install omegaconf datasets pyyaml matplotlib pandas seaborn tqdm -q

# ── Fix: __init__.py mínimo (evita imports de modelos que precisam de deps extras) ──
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write(
        "from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel\n"
        "from easy_tpp.model.torch_model.torch_thp    import THP    as TorchTHP\n"
        "from easy_tpp.model.torch_model.torch_rothp  import RoTHP  as TorchRoTHP\n"
        "from easy_tpp.model.torch_model.torch_hothp  import HoTHP  as TorchHoTHP\n"
    )

# ── Fix: força import de todos os modelos no runner ───────────────────
# generate_model_from_config() usa __subclasses__() — só encontra classes
# cujos módulos foram importados. Sem este fix: RuntimeError: No model named RoTHP
_runner_path = 'ufc-easytpp/easy_tpp/runner/tpp_runner.py'
with open(_runner_path, 'r') as f:
    _content = f.read()
if 'import easy_tpp.model' not in _content:
    _content = _content.replace(
        'from collections import OrderedDict\n',
        'from collections import OrderedDict\nimport easy_tpp.model  # noqa: F401\n'
    )
    with open(_runner_path, 'w') as f:
        f.write(_content)

import torch
GPU = 0 if torch.cuda.is_available() else -1
print(f'OK — GPU={GPU}, torch={torch.__version__}')

Cloning into 'ufc-easytpp'...
remote: Enumerating objects: 702, done.
remote: Counting objects: 100% (702/702), done.
remote: Compressing objects: 100% (524/524), done.
remote: Total 702 (delta 205), reused 648 (delta 151), pack-reused 0 (from 0)
Receiving objects: 100% (702/702), 33.32 MiB | 8.62 MiB/s, done.
Resolving deltas: 100% (205/205), done.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
OK — GPU=-1, torch=2.2.2


In [4]:
import gc
import pickle
import tempfile
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

from easy_tpp.config_factory import Config
from easy_tpp.runner import Runner

sns.set_theme(style='whitegrid')

# ── Parâmetros do experimento ─────────────────────────────────────────
SEEDS          = [2019, 2020, 2021]
MODELS         = ['THP', 'RoTHP', 'HoTHP']
MAX_EPOCH      = 100
LEARNING_RATE  = 1e-3
BATCH_SIZE     = 256
HIDDEN_SIZE    = 64
NUM_HEADS      = 2
NUM_LAYERS     = 2
DROPOUT        = 0.1
TIME_EMB_SIZE  = 16

# ── Datasets com comprimentos reais das sequências ────────────────────
# Levantamento feito com datasets.load_dataset('easytpp/<name>'):
#
#   taxi:          min=36  median=38  max=38    → seqs uniformes, sem espaço p/ extrapolar
#   retweet:       min=50  median=90  max=264   → extrapola bem
#   earthquake:    min=15  median=16  max=18    → seqs muito curtas
#   stackoverflow: min=41  median=57  max=101   → extrapola
#   amazon:        min=14  median=42  max=94    → extrapola
#
# Estratégia de extrapolação:
#   - train_max_len: tamanho curto para treino (trunca sequências longas)
#   - eval_1x:       mesmo tamanho do treino (sem extrapolação)
#   - eval_5x:       5× o tamanho do treino (extrapolação real)
#   - Para taxi e earthquake: sequências são muito curtas, max_len não trunca nada.
#     Usamos o tamanho natural e reportamos sem extrapolação.

DATASETS = {
    # 'retweet': {
    #     'num_event_types': 3,
    #     'pad_token_id':    3,
    #     'train_max_len':   50,      # treina com seqs truncadas em 50
    #     'eval_1x':         50,      # avalia truncado (mesma condição do treino)
    #     'eval_5x':         250,     # avalia com 5× (max real = 264)
    # },
    # 'taxi': {
    #     'num_event_types': 10,
    #     'pad_token_id':    10,
    #     'train_max_len':   38,      # max real = 38, não há o que truncar
    #     'eval_1x':         38,
    #     'eval_5x':         None,    # None = sem extrapolação (seqs curtas demais)
    # },
    # 'earthquake': {
    #     'num_event_types': 7,
    #     'pad_token_id':    7,
    #     'train_max_len':   18,      # max real = 18
    #     'eval_1x':         18,
    #     'eval_5x':         None,    # sem extrapolação
    # },
    'stackoverflow': {
        'num_event_types': 22,
        'pad_token_id':    22,
        'train_max_len':   20,      # treina com 20 (median=57, max=101)
        'eval_1x':         20,
        'eval_5x':         100,     # 5× (max real = 101)
    },
    'amazon': {
        'num_event_types': 16,
        'pad_token_id':    16,
        'train_max_len':   18,      # treina com 18 (median=42, max=94)
        'eval_1x':         18,
        'eval_5x':         90,      # 5× (max real = 94)
    },
}

CHECKPOINT_FILE = 'benchmark_progress.pkl'

def free_gpu():
    """Libera memória GPU acumulada entre runs."""
    gc.collect()
    torch.cuda.empty_cache()

print('Configuração dos datasets:')
print(f'{"Dataset":<16} {"train":>6} {"1x":>6} {"5x":>6}  notas')
print('-' * 65)
for name, cfg in DATASETS.items():
    ext = cfg['eval_5x'] if cfg['eval_5x'] else '  —'
    nota = '' if cfg['eval_5x'] else '(seqs curtas, sem extrapolação)'
    print(f'{name:<16} {cfg["train_max_len"]:>6} {cfg["eval_1x"]:>6} {str(ext):>6}  {nota}')

print(f'\nModels: {MODELS}')
print(f'Seeds: {SEEDS}')

Configuração dos datasets:
Dataset           train     1x     5x  notas
-----------------------------------------------------------------
stackoverflow        20     20    100  
amazon               18     18     90  

Models: ['THP', 'RoTHP', 'HoTHP']
Seeds: [2019, 2020, 2021]


In [5]:
# ── Funções auxiliares ────────────────────────────────────────────────

def load_progress():
    """Carrega resultados parciais do pickle."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            data = pickle.load(f)
        print(f'Progresso carregado: {len(data["results"])} resultados de eval, '
              f'{len(data["trained"])} modelos treinados')
        return data
    return {'results': [], 'trained': set()}


def save_progress(progress):
    """Salva resultados parciais no pickle."""
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(progress, f)


def make_yaml(dataset_name, model_id, seed, max_len, stage='train',
              pretrained_model_dir=None):
    """Gera config dict YAML para treino ou eval."""
    ds = DATASETS[dataset_name]
    exp_id = f'{model_id}_{stage}'

    model_cfg = {
        'hidden_size':    HIDDEN_SIZE,
        'num_heads':      NUM_HEADS,
        'num_layers':     NUM_LAYERS,
        'dropout':        DROPOUT,
        'time_emb_size':  TIME_EMB_SIZE,
        'use_ln':         False,
        'thinning': {
            'num_sample': 1, 'num_exp': 500, 'look_ahead_time': 10,
            'patience_counter': 5, 'over_sample_rate': 5,
            'num_samples_boundary': 5, 'dtime_max': 10,
            'num_seq': 10, 'num_step_gen': 1,
        },
    }

    if stage == 'train':
        model_cfg['loss_integral_num_sample_per_step'] = 20
        model_cfg['mc_num_sample_per_step'] = 20
    if pretrained_model_dir:
        model_cfg['pretrained_model_dir'] = pretrained_model_dir

    trainer_cfg = {
        'batch_size': BATCH_SIZE,
        'max_epoch':  MAX_EPOCH if stage == 'train' else 1,
        'seed':       seed,
        'gpu':        GPU,
        'metrics':    ['acc', 'rmse'],
    }
    if stage == 'train':
        trainer_cfg.update({
            'valid_freq': 1, 'use_tfb': False,
            'optimizer': 'adam', 'learning_rate': LEARNING_RATE,
            'shuffle': False,
        })

    config = {
        'pipeline_config_id': 'runner_config',
        'data': {
            dataset_name: {
                'data_format': 'json',
                'train_dir': f'easytpp/{dataset_name}',
                'valid_dir': f'easytpp/{dataset_name}',
                'test_dir':  f'easytpp/{dataset_name}',
                'data_specs': {
                    'num_event_types': ds['num_event_types'],
                    'pad_token_id':    ds['pad_token_id'],
                    'padding_side':    'right',
                    'truncation_side': 'right',
                    'max_len':         max_len,
                },
            },
        },
        exp_id: {
            'base_config': {
                'stage':      stage,
                'backend':    'torch',
                'dataset_id': dataset_name,
                'runner_id':  'std_tpp',
                'model_id':   model_id,
                'base_dir':   f'./checkpoints/{dataset_name}/{model_id}/seed{seed}/',
            },
            'trainer_config': trainer_cfg,
            'model_config':  model_cfg,
        },
    }
    return config, exp_id


def write_yaml_and_load(config_dict, experiment_id):
    """Escreve YAML em arquivo temporário e retorna Config."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(config_dict, f, default_flow_style=False)
        tmp_path = f.name
    try:
        cfg = Config.build_from_yaml_file(tmp_path, experiment_id=experiment_id)
    finally:
        os.unlink(tmp_path)
    return cfg


def get_model_dir(ds_name, model_id, seed):
    """Retorna o caminho real do checkpoint salvo pelo EasyTPP.

    EasyTPP salva em {base_dir}/{unique_id}/models/saved_model,
    onde {unique_id} é gerado automaticamente por get_unique_id().
    Esta função escaneia a base_dir para encontrar o checkpoint mais recente.
    """
    base = f'./checkpoints/{ds_name}/{model_id}/seed{seed}'
    if not os.path.isdir(base):
        return f'{base}/models/saved_model'  # placeholder que nao existe
    candidates = []
    for entry in os.scandir(base):
        if entry.is_dir():
            candidate = os.path.join(entry.path, 'models', 'saved_model')
            if os.path.exists(candidate):
                candidates.append((entry.stat().st_mtime, candidate))
    if candidates:
        return sorted(candidates)[-1][1]  # mais recente
    return f'{base}/models/saved_model'   # placeholder que nao existe


print('Funções auxiliares definidas.')

Funções auxiliares definidas.


In [ ]:
# ── Treinamento ──────────────────────────────────────────────────────
# Treina cada (dataset, modelo, seed) com train_max_len.
# O modelo treinado será reutilizado na avaliação com 1× e 5×.
#
# IMPORTANTE: verifica se o checkpoint físico ainda existe no disco.
# Após um reset de sessão do Colab, o pickle persiste (Google Drive)
# mas os checkpoints são perdidos — sem esta verificação, todos os
# modelos seriam pulados e as avaliações falhariam silenciosamente.

progress = load_progress()

total_train = len(DATASETS) * len(MODELS) * len(SEEDS)
done_train = len(progress['trained'])
print(f'Treinamentos: {done_train}/{total_train} já concluídos\n')

for ds_name, ds_cfg in DATASETS.items():
    for model_id in MODELS:
        for seed in SEEDS:
            key = (ds_name, model_id, seed)
            if key in progress['trained']:
                # Verifica se o checkpoint físico ainda existe no disco.
                # Se o Colab resetou, o pickle sobrevive mas os arquivos somem.
                model_dir = get_model_dir(ds_name, model_id, seed)
                if os.path.exists(model_dir):
                    print(f'  [SKIP] {ds_name}/{model_id}/seed{seed}')
                    continue
                else:
                    print(f'  [RETRAIN] {ds_name}/{model_id}/seed{seed} '
                          f'— checkpoint perdido após reset, re-treinando...')
                    progress['trained'].discard(key)
                    # Invalida evals antigas deste modelo (serão re-executadas)
                    progress['results'] = [
                        r for r in progress['results']
                        if not (r['dataset'] == ds_name and
                                r['model'] == model_id and
                                r['seed'] == seed)
                    ]
                    save_progress(progress)

            train_len = ds_cfg['train_max_len']
            print(f'\n{"="*60}')
            print(f'  {ds_name} / {model_id} / seed={seed}  [max_len={train_len}]')
            print(f'{"="*60}')

            cfg_dict, exp_id = make_yaml(ds_name, model_id, seed,
                                         max_len=train_len, stage='train')
            cfg = write_yaml_and_load(cfg_dict, experiment_id=exp_id)
            runner = Runner.build_from_config(cfg)
            runner.run()
            del runner
            free_gpu()

            progress['trained'].add(key)
            save_progress(progress)
            print(f'  [OK] salvo em {get_model_dir(ds_name, model_id, seed)}')

print(f'\nTreinamento concluído! {len(progress["trained"])}/{total_train}')

Progresso carregado: 0 resultados de eval, 24 modelos treinados
Treinamentos: 24/18 já concluídos

  [RETRAIN] stackoverflow/THP/seed2019 — checkpoint perdido após reset, re-treinando...

  stackoverflow / THP / seed=2019  [max_len=20]
2026-05-18 09:01:50,895 - config.py[pid:10526;line:34:build_from_yaml_file] - CRITICAL: Load pipeline config class RunnerConfig
2026-05-18 09:01:50,898 - runner_config.py[pid:10526;line:140:update_config] - CRITICAL: train model THP using CPU with torch backend
2026-05-18 09:01:50,912 - runner_config.py[pid:10526;line:35:__init__] - INFO: Save the config to ./checkpoints/stackoverflow/THP/seed2019/10526_140704643678464_260518-090150/THP_train_output.yaml
2026-05-18 09:01:50,913 - base_runner.py[pid:10526;line:176:save_log] - INFO: Save the log to ./checkpoints/stackoverflow/THP/seed2019/10526_140704643678464_260518-090150/log
0.8889953323541242 1.0758269014131785
min_dt: 0.0
max_dt: 20.337890625


2026-05-18 09:02:13.049787: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-05-18 09:02:29,618 - tpp_runner.py[pid:10526;line:61:_init_model] - INFO: Num of model parameters 44504
2026-05-18 09:02:34,227 - base_runner.py[pid:10526;line:98:train] - INFO: Data 'stackoverflow' loaded...
2026-05-18 09:02:34,228 - base_runner.py[pid:10526;line:103:train] - INFO: Start THP training...
2026-05-18 09:02:38,996 - tpp_runner.py[pid:10526;line:97:_train_model] - INFO: [ Epoch 0 (train) ]: train loglike is -12.392580317157897, num_events is 89096
2026-05-18 09:02:48,731 - tpp_runner.py[pid:10526;line:108:_train_model] - INFO: [ Epoch 0 (valid) ]:  valid loglike is -10.893382063010133, num_events is 25361, acc is 0.053901660029178664, rmse is 1.3607013880839056
2026-05-18 09:02:57,726 - tpp_runner.py[pid:10526;line:123:_train_model] - INFO: [ Epoch 0 (test) ]: test loglike is -10.545478340257304, num_events is 26117, acc is 0.04908680170004212, rmse is 1.3252138535154825
2026-05-18 09:02:57,727 - tpp_runner.py[pid:10526;line:125:_train_model] - CRITICAL: current best 

In [7]:
# ── Avaliação ─────────────────────────────────────────────────────────
# Para cada modelo treinado:
#   - Avalia com eval_1x (mesma condição do treino)
#   - Avalia com eval_5x (extrapolação) — somente se o dataset suportar

progress = load_progress()

done_evals = set()
for r in progress['results']:
    done_evals.add((r['dataset'], r['model'], r['seed'], r['extrap']))

# Contar total de evals
total_evals = 0
for ds_cfg in DATASETS.values():
    n_scenarios = 1 + (1 if ds_cfg['eval_5x'] else 0)
    total_evals += len(MODELS) * len(SEEDS) * n_scenarios
print(f'Avaliações: {len(done_evals)}/{total_evals} já concluídas\n')

for ds_name, ds_cfg in DATASETS.items():
    # Montar cenários de avaliação
    scenarios = [('1x', ds_cfg['eval_1x'])]
    if ds_cfg['eval_5x']:
        scenarios.append(('5x', ds_cfg['eval_5x']))

    for model_id in MODELS:
        for seed in SEEDS:
            model_dir = get_model_dir(ds_name, model_id, seed)
            if not os.path.exists(model_dir):
                print(f'  [WARN] Checkpoint não encontrado: {model_dir}')
                continue

            for extrap_name, max_len in scenarios:
                eval_key = (ds_name, model_id, seed, extrap_name)
                if eval_key in done_evals:
                    print(f'  [SKIP] {ds_name}/{model_id}/seed{seed}/{extrap_name}')
                    continue

                print(f'  Eval: {ds_name}/{model_id}/seed{seed}'
                      f' [{extrap_name}, max_len={max_len}]', end=' → ')

                cfg_dict, exp_id = make_yaml(
                    ds_name, model_id, seed, max_len=max_len,
                    stage='eval', pretrained_model_dir=model_dir)
                cfg = write_yaml_and_load(cfg_dict, experiment_id=exp_id)
                runner = Runner.build_from_config(cfg)

                test_loader = runner._data_loader.test_loader()
                metrics = runner._evaluate_model(test_loader)
                del runner
                free_gpu()

                result = {
                    'dataset':  ds_name,
                    'model':    model_id,
                    'seed':     seed,
                    'extrap':   extrap_name,
                    'max_len':  max_len,
                    'nll':      -metrics.get('loglike', float('nan')),
                    'acc':      metrics.get('acc', float('nan')),
                    'rmse':     metrics.get('rmse', float('nan')),
                }
                progress['results'].append(result)
                save_progress(progress)
                print(f'NLL={result["nll"]:.4f}  ACC={result["acc"]:.4f}  RMSE={result["rmse"]:.4f}')

print(f'\nAvaliação concluída! {len(progress["results"])} resultados salvos.')

Progresso carregado: 0 resultados de eval, 24 modelos treinados
Avaliações: 0/36 já concluídas

  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/THP/seed2019/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/THP/seed2020/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/THP/seed2021/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/RoTHP/seed2019/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/RoTHP/seed2020/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/RoTHP/seed2021/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/HoTHP/seed2019/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/HoTHP/seed2020/models/saved_model
  [WARN] Checkpoint não encontrado: ./checkpoints/stackoverflow/HoTHP/seed2021/models/saved_model
  [WARN] Checkpoint não enco

In [ ]:
# ── Carregar resultados e montar DataFrame ───────────────────────────

progress = load_progress()
df = pd.DataFrame(progress['results'])
print(f'Total de resultados: {len(df)}')
display(df.head(10))

# Separar datasets com e sem extrapolação
ds_with_extrap = [name for name, cfg in DATASETS.items() if cfg['eval_5x']]
ds_no_extrap   = [name for name, cfg in DATASETS.items() if not cfg['eval_5x']]
print(f'\nCom extrapolação 5×: {ds_with_extrap}')
print(f'Sem extrapolação:    {ds_no_extrap}')

# Tabela resumo
summary = df.groupby(['dataset', 'model', 'extrap']).agg(
    nll_mean=('nll', 'mean'), nll_std=('nll', 'std'),
    acc_mean=('acc', 'mean'), acc_std=('acc', 'std'),
    rmse_mean=('rmse', 'mean'), rmse_std=('rmse', 'std'),
).reset_index()

for metric in ['nll', 'acc', 'rmse']:
    summary[metric] = summary.apply(
        lambda r: f"{r[f'{metric}_mean']:.4f} \u00b1 {r[f'{metric}_std']:.4f}", axis=1)

print('\n── Resultados Agregados ──')
display(summary[['dataset', 'model', 'extrap', 'nll', 'acc', 'rmse']])

In [ ]:
# ── Gráfico 1: NLL em todos os datasets (1× e 5× onde disponível) ───

COLORS = {'THP': '#4C72B0', 'RoTHP': '#55A868', 'HoTHP': '#C44E52'}
bar_width = 0.25
ds_names = list(DATASETS.keys())
n_ds = len(ds_names)

fig, axes = plt.subplots(1, n_ds, figsize=(4 * n_ds, 5), sharey=False)

for idx, ds_name in enumerate(ds_names):
    ax = axes[idx]
    sub = df[df['dataset'] == ds_name]
    ds_cfg = DATASETS[ds_name]

    # Cenários disponíveis para este dataset
    scenarios = ['1x']
    if ds_cfg['eval_5x']:
        scenarios.append('5x')

    x = np.arange(len(scenarios))

    for mi, model_id in enumerate(MODELS):
        model_sub = sub[sub['model'] == model_id]
        means, stds = [], []
        for ext_name in scenarios:
            ext_sub = model_sub[model_sub['extrap'] == ext_name]
            means.append(ext_sub['nll'].mean() if len(ext_sub) > 0 else 0)
            stds.append(ext_sub['nll'].std() if len(ext_sub) > 0 else 0)

        offset = (mi - 1) * bar_width
        ax.bar(x + offset, means, bar_width,
               yerr=stds, capsize=4,
               label=model_id, color=COLORS[model_id], alpha=0.85)

    ax.set_xticks(x)
    scenario_labels = []
    for s in scenarios:
        ml = ds_cfg[f'eval_{s}']
        scenario_labels.append(f'{s}\n(len={ml})')
    ax.set_xticklabels(scenario_labels, fontsize=9)
    if idx == 0:
        ax.set_ylabel('NLL (menor = melhor)')
    ax.set_title(f'{ds_name}\ntrain_len={ds_cfg["train_max_len"]}',
                 fontweight='bold', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('NLL por Dataset e Cenario de Extrapolacao\n'
             f'({len(SEEDS)} seeds, barras = media +/- std)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('benchmark_nll.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: benchmark_nll.png')

In [ ]:
# ── Gráfico 2: Degradação NLL (5× - 1×) — somente datasets com extrapolação

if len(ds_with_extrap) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))

    degradation_data = []
    for ds_name in ds_with_extrap:
        for model_id in MODELS:
            for seed in SEEDS:
                sub = df[(df['dataset'] == ds_name) &
                         (df['model'] == model_id) &
                         (df['seed'] == seed)]
                nll_1x = sub[sub['extrap'] == '1x']['nll'].values
                nll_5x = sub[sub['extrap'] == '5x']['nll'].values
                if len(nll_1x) > 0 and len(nll_5x) > 0:
                    degradation_data.append({
                        'dataset': ds_name,
                        'model': model_id,
                        'seed': seed,
                        'delta_nll': nll_5x[0] - nll_1x[0],
                    })

    df_deg = pd.DataFrame(degradation_data)
    n_ext = len(ds_with_extrap)
    x = np.arange(n_ext)

    for mi, model_id in enumerate(MODELS):
        model_sub = df_deg[df_deg['model'] == model_id]
        means = [model_sub[model_sub['dataset'] == ds]['delta_nll'].mean()
                 for ds in ds_with_extrap]
        stds = [model_sub[model_sub['dataset'] == ds]['delta_nll'].std()
                for ds in ds_with_extrap]
        offset = (mi - 1) * bar_width
        ax.bar(x + offset, means, bar_width,
               yerr=stds, capsize=4,
               label=model_id, color=COLORS[model_id], alpha=0.85)

    ax.axhline(0, color='gray', ls='--', lw=1.2)

    # Anotar train_len → eval_5x por dataset
    labels = [f'{ds}\n({DATASETS[ds]["train_max_len"]}→{DATASETS[ds]["eval_5x"]})'
              for ds in ds_with_extrap]
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel('Dataset (train_len → eval_5x_len)')
    ax.set_ylabel('$\\Delta$NLL = NLL(5x) - NLL(1x)')
    ax.set_title('Degradacao com Extrapolacao 5x\n'
                 '(positivo = piorou, negativo = melhorou)',
                 fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('benchmark_degradation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvo: benchmark_degradation.png')
else:
    print('Nenhum dataset com extrapolação — pulando gráfico de degradação.')

In [ ]:
# ── Gráfico 3: Accuracy e RMSE ──────────────────────────────────────

fig, axes = plt.subplots(2, n_ds, figsize=(4 * n_ds, 8), sharey=False)

for idx, ds_name in enumerate(ds_names):
    sub = df[df['dataset'] == ds_name]
    ds_cfg = DATASETS[ds_name]

    scenarios = ['1x']
    if ds_cfg['eval_5x']:
        scenarios.append('5x')
    x = np.arange(len(scenarios))

    for row, (metric, ylabel) in enumerate([
        ('acc', 'Accuracy (maior = melhor)'),
        ('rmse', 'RMSE (menor = melhor)'),
    ]):
        ax = axes[row][idx]
        for mi, model_id in enumerate(MODELS):
            model_sub = sub[sub['model'] == model_id]
            means, stds = [], []
            for ext_name in scenarios:
                ext_sub = model_sub[model_sub['extrap'] == ext_name]
                means.append(ext_sub[metric].mean() if len(ext_sub) > 0 else 0)
                stds.append(ext_sub[metric].std() if len(ext_sub) > 0 else 0)
            offset = (mi - 1) * bar_width
            ax.bar(x + offset, means, bar_width,
                   yerr=stds, capsize=3,
                   label=model_id, color=COLORS[model_id], alpha=0.85)

        scenario_labels = [f'{s}\n(len={ds_cfg[f"eval_{s}"]})'
                           for s in scenarios if ds_cfg.get(f'eval_{s}')]
        ax.set_xticks(x)
        ax.set_xticklabels(scenario_labels, fontsize=9)
        if idx == 0:
            ax.set_ylabel(ylabel)
        ax.set_title(f'{ds_name}', fontsize=10, fontweight='bold')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Accuracy (tipo) e RMSE (tempo)\n'
             f'({len(SEEDS)} seeds)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('benchmark_acc_rmse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: benchmark_acc_rmse.png')

In [ ]:
# ── Tabela final ─────────────────────────────────────────────────────

print('=' * 90)
print('TABELA FINAL — BENCHMARK REAL DATASETS')
print('=' * 90)

ds_names = list(DATASETS.keys())

for extrap_name in ['1x', '5x']:
    # Quais datasets têm esse cenário
    ds_for_this = [name for name in ds_names
                   if extrap_name == '1x' or DATASETS[name]['eval_5x']]
    if not ds_for_this:
        continue

    print(f'\n── {extrap_name} ──')
    print(f'{"Dataset":<16} {"Model":<8} {"max_len":>7} '
          f'{"NLL":>16} {"ACC":>16} {"RMSE":>16}')
    print('-' * 85)

    for ds_name in ds_for_this:
        ds_cfg = DATASETS[ds_name]
        eval_len = ds_cfg.get(f'eval_{extrap_name}')
        if eval_len is None:
            continue

        # Encontrar melhor NLL para marcar
        best_nll = float('inf')
        for model_id in MODELS:
            sub = df[(df['dataset'] == ds_name) &
                     (df['model'] == model_id) &
                     (df['extrap'] == extrap_name)]
            if len(sub) > 0 and sub['nll'].mean() < best_nll:
                best_nll = sub['nll'].mean()

        for model_id in MODELS:
            sub = df[(df['dataset'] == ds_name) &
                     (df['model'] == model_id) &
                     (df['extrap'] == extrap_name)]
            if len(sub) == 0:
                continue
            nll_m, nll_s = sub['nll'].mean(), sub['nll'].std()
            acc_m, acc_s = sub['acc'].mean(), sub['acc'].std()
            rmse_m, rmse_s = sub['rmse'].mean(), sub['rmse'].std()

            marker = ' *' if abs(nll_m - best_nll) < 1e-6 else '  '
            ds_label = ds_name if model_id == MODELS[0] else ''
            print(f'{ds_label:<16} {model_id:<8} {eval_len:>7} '
                  f'{nll_m:>7.4f}+/-{nll_s:.4f}{marker} '
                  f'{acc_m:>7.4f}+/-{acc_s:.4f}  '
                  f'{rmse_m:>7.4f}+/-{rmse_s:.4f}')

print('\n* = melhor NLL no dataset/cenario')

In [ ]:
# ── Gráfico 4: Comparação direta THP vs RoTHP vs HoTHP (1× only) ───
# Todos os 5 datasets lado a lado, 3 métricas.

df_1x = df[df['extrap'] == '1x']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax_idx, (metric, title, invert) in enumerate([
    ('nll',  'NLL (menor = melhor)',      True),
    ('acc',  'Accuracy (maior = melhor)', False),
    ('rmse', 'RMSE (menor = melhor)',     True),
]):
    ax = axes[ax_idx]
    x = np.arange(n_ds)

    for mi, model_id in enumerate(MODELS):
        model_sub = df_1x[df_1x['model'] == model_id]
        means = [model_sub[model_sub['dataset'] == ds][metric].mean()
                 for ds in ds_names]
        stds = [model_sub[model_sub['dataset'] == ds][metric].std()
                for ds in ds_names]
        offset = (mi - 1) * bar_width
        ax.bar(x + offset, means, bar_width,
               yerr=stds, capsize=3,
               label=model_id, color=COLORS[model_id], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(ds_names, fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Comparacao THP vs RoTHP vs HoTHP — Cenario 1x (sem extrapolacao)\n'
             f'{len(SEEDS)} seeds',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('benchmark_1x_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: benchmark_1x_comparison.png')